# FIN pretrained on MotionSense + real + warped windows vs random-init, 7 WISDM training users

- **train:** WISDM users 12, 18, 19, 20, 31, 34, 36, **test:** the other 29
- **pretrained, frozen:** the FIN imitates spectral entropy, autocorrelation peak and spectral peak on single-axis windows from all of MotionSense (24 other people, labels not used), the 7 training users' real windows and 12,006 warped copies of them; then it stays fixed and only the head learns
- **random-init, trainable:** same architecture, random weights

In [ ]:
import os
import sys
import warnings

os.environ["KERAS_BACKEND"] = "torch"
warnings.filterwarnings("ignore")
FIN = os.path.dirname(os.getcwd())
sys.path[:0] = [FIN, os.path.join(FIN, "FIN-classifiers"), os.path.join(FIN, "data-processing")]

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from keras import layers, ops
from sklearn.metrics import f1_score

import config as cfg
import wisdm_windowing as ww
from augmentation import warped
import motionsense_windowing as ms
from bilin_fin_temporal_plus_original import RhythmFIN, targets

SEED = 0
TRAIN_USERS = [12, 18, 19, 20, 31, 34, 36]
N_WARPED = 12006

## Data

In [ ]:
windows, users, labels = ww.window_raw()
print(f"{len(windows)} windows, {len(np.unique(users))} users")

In [ ]:
tr = np.isin(users, TRAIN_USERS)
te = ~tr
real = windows[tr].reshape(-1, cfg.SEGMENT)  # every axis of every training window
rng = np.random.default_rng(SEED)
warp = np.stack([warped(rng, real) for _ in range(N_WARPED)]).astype(np.float32)
pool = np.concatenate([ms.window_raw()[0].reshape(-1, cfg.SEGMENT), real, warp])
print(f"test {te.sum()} windows of {36 - len(TRAIN_USERS)} users; pretraining on {len(pool)} single-axis windows")

## Model

In [ ]:
class TestAccuracy(keras.callbacks.Callback):
    #Test accuracy after every epoch

    def __init__(self, x, y):
        super().__init__()
        self.x, self.y, self.curve = x, y, []

    def on_epoch_end(self, epoch, logs=None):
        rng = torch.random.get_rng_state()
        pred = self.model.predict(self.x, batch_size=1024, verbose=0).argmax(axis=1)
        torch.random.set_rng_state(rng)
        self.curve.append(float(np.mean(pred == self.y)))


class FINClassifier(keras.Model):

    def __init__(self, fin, **kw):
        super().__init__(**kw)
        self.fin = fin
        self.hidden = layers.Dense(cfg.HEAD_WIDTH, activation="relu")
        self.out = layers.Dense(cfg.N_CLASSES, activation="softmax")

    def call(self, x):
        h = ops.concatenate([self.fin.embed(x[:, i, :]) for i in range(len(cfg.AXES))], axis=1)
        return self.out(self.hidden(h))


def pretrain(axes, seed):
    keras.utils.set_random_seed(seed)
    fin = RhythmFIN()
    y = targets(axes)
    fin.compile(optimizer=keras.optimizers.Adam(cfg.FIN_LR), loss="mse")
    fin.fit(axes, ((y - y.mean(0)) / y.std(0)).astype(np.float32),
            epochs=cfg.FIN_EPOCHS, batch_size=cfg.BATCH_SIZE, verbose=0)
    return fin.get_weights()


def classify(tr, te, seed, weights, trainable):
    keras.utils.set_random_seed(seed)
    fin = RhythmFIN()
    if weights is not None:
        fin(np.zeros((1, cfg.SEGMENT), np.float32))
        fin.set_weights(weights)
    fin.trainable = trainable
    keras.utils.set_random_seed(seed)
    model = FINClassifier(fin)
    track = TestAccuracy(windows[te], labels[te])
    model.compile(optimizer=keras.optimizers.Adam(cfg.HEAD_LR), loss="sparse_categorical_crossentropy")
    model.fit(windows[tr], labels[tr], epochs=cfg.HEAD_EPOCHS, batch_size=cfg.BATCH_SIZE,
              verbose=0, callbacks=[track])
    pred = model.predict(windows[te], batch_size=1024, verbose=0).argmax(axis=1)
    return pred, track.curve

## Experiment

In [ ]:
weights = pretrain(pool, SEED)

MODELS = {"pretrained on MotionSense + real + warped, frozen": (weights, False), "random-init, trainable": (None, True)}
COLORS = dict(zip(MODELS, ["#2a78d6", "#eb6834"]))
rows, curves = [], {}
for model, (w, trainable) in MODELS.items():
    pred, curves[model] = classify(tr, te, SEED, w, trainable)
    truth = labels[te]
    rows.append({"model": model, "accuracy": np.mean(pred == truth), "macro F1": f1_score(truth, pred, average="macro"),
                 **{a: np.mean(pred[truth == i] == i) for i, a in enumerate(cfg.ACTIVITIES)}})
pd.DataFrame(rows).set_index("model").round(3)

In [ ]:
epochs = np.arange(1, cfg.HEAD_EPOCHS + 1)
fig, ax = plt.subplots(figsize=(7, 4))
for model, color in COLORS.items():
    ax.plot(epochs, curves[model], color=color, linewidth=1.8, label=model)
    ax.annotate(f"{curves[model][-1]:.3f}", (epochs[-1], curves[model][-1]), xytext=(5, 0),
                textcoords="offset points", va="center", fontsize=9)
ax.set_xlabel("epoch")
ax.set_ylabel(f"test accuracy, {36 - len(TRAIN_USERS)} unseen users")
ax.set_xlim(1, cfg.HEAD_EPOCHS + 4)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e1e0d9", linewidth=0.6)
ax.legend(frameon=False, loc="lower right")
ax.set_title(f"WISDM training users {TRAIN_USERS}, seed {SEED}", fontsize=10, loc="left")
plt.tight_layout()
plt.show()